# Countdown × GRPO — Qwen3.5-0.8B-Base, no SFT

Runner for `countdown/GUIDE.md`. Read the guide; this notebook just executes it.

**Runtime → A100.** Expect a full 400-step run to take a few hours; the first 5 steps tell you the real `step_time`.

In [ ]:
!nvidia-smi
import torch; print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

In [ ]:
%%capture
!pip install -U "trl>=1.11" "transformers>=5.2" datasets peft accelerate wandb matplotlib
!pip install flash-linear-attention          # Qwen3.5 Gated-DeltaNet kernels (Triton). Without it: silent slow path.
!pip install wordfreq
# Optional & minor (often fails to build on Colab; skip if so): !pip install causal-conv1d --no-build-isolation
!python -c "from fla.ops.gated_delta_rule import chunk_gated_delta_rule; print('fla OK')"   # must print fla OK
# Optional fast generation:     !pip install "vllm==0.27.1"   -> then add --use_vllm to the train command

In [ ]:
# Get the code. EITHER clone your repo ...
REPO_URL = "https://github.com/<you>/grpo_puzzles"   # <- fill in after you push this folder
# !git clone $REPO_URL
# ... OR upload the .py files from this folder with the Colab file browser into /content/grpo_puzzles/

In [ ]:
from huggingface_hub import login; login()   # push results later
import wandb; wandb.login()                        # the curves are the deliverable

In [ ]:
%cd /content/grpo_puzzles/countdown
!python rewards.py | tail -3
!python data.py | head -5

## Step 3 — baseline (go / no-go). Read `results/base-pass8.json` → `pass@8`. See the decision table in GUIDE.md §5.

In [ ]:
!python eval_countdown.py --model Qwen/Qwen3.5-0.8B-Base --tag base-greedy
!python eval_countdown.py --model Qwen/Qwen3.5-0.8B-Base --tag base-pass8 --sample --k 8 --n 200

## Step 4 — train
First 5 steps: check `step_time` and `frac_reward_zero_std`.

In [ ]:
RUN_DIR = "runs/countdown-qwen3.5-0.8b-base"
# thin signal at baseline?  add:  --curriculum 3only --max_steps 150   (then resume with --curriculum none)
# L4 24 GB?                 add:  --per_device_train_batch_size 4 --grad_accum 8     or  --lora --lr 2e-5
!python train_countdown_grpo.py --output_dir $RUN_DIR --max_steps 400 --report_to wandb --run_name cd-0.8b-base

## Step 5 — curves (from the checkpoint's `trainer_state.json`; W&B has the full set)

In [ ]:
KEYS = ["rewards/answer_reward/mean", "rewards/format_reward/mean", "completions/mean_length"]
TITLES = ["solve rate (train prompts)", "format reward", "completion length"]
import json, glob, os
import matplotlib.pyplot as plt

def latest_state(run_dir):
    ck = sorted(glob.glob(f"{run_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    path = os.path.join(ck[-1], "trainer_state.json") if ck else os.path.join(run_dir, "final", "trainer_state.json")
    return json.load(open(path))["log_history"]

hist = latest_state(RUN_DIR)
def series(key):
    return [(h["step"], h[key]) for h in hist if key in h]

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, key, title in zip(axes, KEYS, TITLES):
    s = series(key)
    if s:
        ax.plot(*zip(*s)); ax.set_title(title); ax.set_xlabel("optimizer step"); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RUN_DIR}/curves.png", dpi=150); plt.show()
print("saved", f"{RUN_DIR}/curves.png")

## Step 6 — after: same 500 held-out puzzles, before vs after

In [ ]:
!python eval_countdown.py --model $RUN_DIR/final --tag grpo400
!python eval_countdown.py --model $RUN_DIR/final --tag grpo400-pass8 --sample --k 8 --n 200
import json
for tag in ["base-greedy", "grpo400"]:
    m = json.load(open(f"results/{tag}.json"))["metrics"]
    print(f"{tag:12s} solve={m['solve_rate']:.3f}  3nums={m['solve_rate_3nums']:.3f}  4nums={m['solve_rate_4nums']:.3f}  clean_format={m['format_rate_clean']:.3f}  len={m['mean_completion_tokens']:.0f}")

In [ ]:
# look at what it learned to say
import json
for s in json.load(open("results/grpo400.json"))["samples"][:5]:
    print(s["nums"], "->", s["target"], "| solved:", s["solved"]); print(s["completion"][:600]); print("-"*80)

## Publish
```python
# from transformers import AutoModelForCausalLM, AutoTokenizer
# AutoModelForCausalLM.from_pretrained(f"{RUN_DIR}/final").push_to_hub("<you>/qwen3.5-0.8b-countdown-grpo")
# AutoTokenizer.from_pretrained(f"{RUN_DIR}/final").push_to_hub("<you>/qwen3.5-0.8b-countdown-grpo")
```
Model card: eval protocol (500 held-out, greedy), before/after table, the curve, 3 samples, exact command, cost.